# 🔄 Day 03a: Training Loop — The Complete Recipe

---

## 🎯 What You'll Master Today
- Dataset & DataLoader
- Full training loop with validation
- Early stopping & LR scheduling
- Metrics tracking

**Goal:** Write a production-grade training loop from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  TRAINING LOOP — The Recipe                                      ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  for epoch in range(epochs):                                     ║
║      model.train()                                               ║
║      for batch in train_loader:                                  ║
║          optimizer.zero_grad()                                   ║
║          output = model(batch_x)                                 ║
║          loss = criterion(output, batch_y)                       ║
║          loss.backward()                                         ║
║          optimizer.step()                                        ║
║                                                                   ║
║      model.eval()                                                ║
║      with torch.no_grad():                                       ║
║          val_loss = validate(model, val_loader)                  ║
║                                                                   ║
║      scheduler.step()                                            ║
║      early_stop_check(val_loss)                                  ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np

# ============================================================
# 1. Custom Dataset
# ============================================================

class TabularDataset(Dataset):
    """Custom Dataset for tabular data."""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Generate fake binary classification data
np.random.seed(42)
n_samples = 1000
X = np.random.randn(n_samples, 10).astype(np.float32)
w_true = np.random.randn(10)
y = (X @ w_true > 0).astype(np.float32).reshape(-1, 1)

dataset = TabularDataset(X, y)
print(f"Dataset size: {len(dataset)}")
print(f"Sample: X={dataset[0][0].shape}, y={dataset[0][1].shape}")

# Split: 80% train, 10% val, 10% test
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print(f"\nTrain: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
print(f"Train batches: {len(train_loader)}, batch_size=32")

In [ ]:
# ============================================================
# 2. Model + Loss + Optimizer Setup
# ============================================================

class BinaryNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )
    
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BinaryNet(input_dim=10).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

print(f"Device: {device}")
print(f"Model params: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# ============================================================
# 3. Full Training Loop — MEMORIZE THIS
# ============================================================

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.should_stop = False
    
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(X_batch)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == y_batch).sum().item()
        total += len(y_batch)
    
    return total_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            
            total_loss += loss.item() * len(X_batch)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == y_batch).sum().item()
            total += len(y_batch)
    
    return total_loss / total, correct / total

# --- Training ---
epochs = 50
early_stopping = EarlyStopping(patience=10)
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    scheduler.step(val_loss)
    early_stopping(val_loss)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    if epoch % 10 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d} | Train: loss={train_loss:.4f} acc={train_acc:.4f} | "
              f"Val: loss={val_loss:.4f} acc={val_acc:.4f} | LR: {lr:.6f}")
    
    if early_stopping.should_stop:
        print(f"\n⏹️ Early stopping at epoch {epoch}")
        break

# Final test
test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"\n🏁 Test: loss={test_loss:.4f}, acc={test_acc:.4f}")

In [ ]:
# ============================================================
# 4. LR Schedulers — The Important Ones
# ============================================================

# For demo purposes, show LR decay patterns
model_demo = nn.Linear(10, 1)

schedulers = {
    'StepLR': lambda opt: optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5),
    'CosineAnnealing': lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50),
    'ReduceOnPlateau': lambda opt: optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5),
}

print("LR Scheduler Comparison:")
for name, sched_fn in schedulers.items():
    opt = optim.Adam(model_demo.parameters(), lr=0.001)
    sched = sched_fn(opt)
    lrs = []
    for epoch in range(50):
        lrs.append(opt.param_groups[0]['lr'])
        if name == 'ReduceOnPlateau':
            sched.step(1.0)  # fake val_loss
        else:
            sched.step()
    print(f"  {name}: {lrs[0]:.6f} → {lrs[24]:.6f} → {lrs[49]:.6f}")

print("\n💡 CosineAnnealing = great default. ReduceOnPlateau = adaptive.")

In [ ]:
# ============================================================
# 5. Training Loop Template — Copy & Adapt
# ============================================================

print("""
PyTorch TRAINING LOOP TEMPLATE
══════════════════════════════

# 1. Dataset & DataLoader
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

# 2. Model, Loss, Optimizer
model = MyModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

# 3. Training Loop
for epoch in range(epochs):
    # --- Train ---
    model.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
    
    # --- Validate ---
    model.eval()
    with torch.no_grad():
        val_loss = sum(criterion(model(X.to(device)), y.to(device)).item()
                       for X, y in val_loader) / len(val_loader)
    
    scheduler.step()
    print(f"Epoch {epoch} | Val Loss: {val_loss:.4f}")

# 4. Save best model
torch.save(model.state_dict(), 'best_model.pth')
""")

---

## 🗺️ Training Loop Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  TRAINING LOOP COMPONENTS                                       ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  DATA:                                                           ║
║    Dataset → __len__, __getitem__                               ║
║    DataLoader → batching, shuffling, num_workers                ║
║    random_split → train/val/test split                          ║
║                                                                   ║
║  TRAINING:                                                       ║
║    model.train() → zero_grad → forward → loss → backward → step║
║                                                                   ║
║  VALIDATION:                                                     ║
║    model.eval() → torch.no_grad() → forward → metrics          ║
║                                                                   ║
║  REGULARIZATION:                                                 ║
║    EarlyStopping → stop when val_loss stops improving           ║
║    LR Scheduler  → reduce LR over time                         ║
║    Gradient clip  → torch.nn.utils.clip_grad_norm_              ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Describe the PyTorch training loop | zero_grad → forward → loss → backward → step. Wrap in epoch/batch loops |
| 2 | Dataset vs DataLoader? | Dataset: __len__ + __getitem__ (indexing). DataLoader: batching, shuffling, parallel loading |
| 3 | Why shuffle training but not validation? | Shuffling prevents learning data order. Val doesn't need it — want deterministic results |
| 4 | What is early stopping? | Stop training when val_loss doesn't improve for N epochs. Prevents overfitting |
| 5 | CosineAnnealing vs ReduceOnPlateau? | Cosine: smooth decay, predictable. Plateau: adaptive, reduces on stagnation |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Dataset: __len__ + __getitem__                       │
## │  • DataLoader: batch, shuffle, num_workers              │
## │  • Train: model.train(), Val: model.eval() + no_grad   │
## │  • EarlyStopping + LR scheduler = essential             │
## │  • Always track train AND val metrics                    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **CNN** — convolutional neural networks for images